In [72]:
! pip install web3


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [73]:
from web3 import Web3
import time
from sqlalchemy import create_engine, text

In [ ]:
from config import ALCHEMY_URL

w3 = Web3(Web3.HTTPProvider(ALCHEMY_URL))
print(w3.is_connected())
print(w3.eth.block_number)

In [75]:
erc20_abi = [
    {
        "anonymous": False,
        "inputs": [
            {"indexed": True, "name": "from", "type": "address"},
            {"indexed": True, "name": "to", "type": "address"},
            {"indexed": False, "name": "value", "type": "uint256"}
        ],
        "name": "Transfer",
        "type": "event"
    }
]

In [76]:
usdc_address = Web3.to_checksum_address('0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48')
contract = w3.eth.contract(address=usdc_address, abi=erc20_abi)

In [77]:
event_filter = contract.events.Transfer.create_filter(from_block='latest')

In [78]:
engine = create_engine("sqlite:///transactions")

def init_db():
    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS transactions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                token_symbol TEXT NOT NULL,
                from_address TEXT NOT NULL,
                to_address TEXT NOT NULL,
                amount REAL NOT NULL,
                block_confirmed INTEGER NOT NULL,
                num_confirmations INTEGER NOT NULL,
                transaction_pending TEXT NOT NULL,
                transaction_hash TEXT NOT NULL UNIQUE
            )
        """))

def save_in_db(from_addr, to_addr, value, block_number, num_confirmations, token_symbl, transaction_pending, transaction_hash):
    with engine.connect() as conn:
        try:
            conn.execute(text("""
                INSERT INTO transactions
                (token_symbol, from_address, to_address, amount, block_confirmed, num_confirmations, transaction_pending, transaction_hash)
                VALUES
                (:token_symbol, :from_addr, :to_addr, :amount, :block_confirmed, :num_confirmations, :pending, :tx_hash)
            """), {
                "token_symbol": token_symbl,
                "from_addr": from_addr,
                "to_addr": to_addr,
                "amount": value,
                "block_confirmed": block_number,
                "num_confirmations": num_confirmations,
                "pending": transaction_pending,
                "tx_hash": transaction_hash
            })
            conn.commit()
            print(f"Saved: {from_addr} -> {to_addr} | {value}")
        except Exception as e:
            print(f"Failed to save transaction {transaction_hash}: {e}")  

    

In [ ]:
# queries the table for pending requests, recalculates confirmations against the
# current head block, and flips them once they clear the confirmation threshold

CONFIRMATION_THRESHOLD = 6

def update_pending_transactions(confirmation_threshold=CONFIRMATION_THRESHOLD):
    latest_block = w3.eth.block_number

    with engine.begin() as conn:
        pending = conn.execute(text("""
            SELECT id, transaction_hash, block_confirmed
            FROM transactions
            WHERE transaction_pending = 'Yes'
        """)).fetchall()

        for row in pending:
            num_confirmations = latest_block - row.block_confirmed
            still_pending = "Yes" if num_confirmations < confirmation_threshold else "No"

            conn.execute(text("""
                UPDATE transactions
                SET num_confirmations = :num_confirmations,
                    transaction_pending = :pending
                WHERE id = :id
            """), {
                "num_confirmations": num_confirmations,
                "pending": still_pending,
                "id": row.id,
            })

            if still_pending == "No":
                print(f"Confirmed {row.transaction_hash} ({num_confirmations} confirmations)")

    return len(pending)


In [ ]:
whale_threshold = 100_000 * 10**6
confirmation_threshold = 6

init_db()

while True:
    new_events = event_filter.get_new_entries()
    print(f"Got {len(new_events)} new events")
    for event in new_events:
        from_addr = event['args']['from']
        to_addr = event['args']['to']
        value = event['args']['value']
        block_number = event['blockNumber']
        tx_hash = event['transactionHash'].hex()
        num_confirmations = w3.eth.block_number - event['blockNumber']
        print(f"value={value}, confirmations={num_confirmations}")
        if value > whale_threshold:
            pending = "Yes" if num_confirmations < confirmation_threshold else "No"
            save_in_db(from_addr, to_addr, value, block_number, num_confirmations, "USDT", pending, tx_hash)

    update_pending_transactions(confirmation_threshold)
    time.sleep(5)
